# Lesson 05 — Grouping with `GROUP BY`

In this lesson:
- Use `GROUP BY` to compute **one aggregate per group** (per studio, per year, per genre)
- Use `HAVING` to filter on group results
- Combine grouping with `ORDER BY` and `LIMIT`

Time: 25–30 minutes.

`GROUP BY` is where SQL goes from "give me a number" to "give me a *report*." It's the muscle behind every dashboard you've ever looked at.

## Setup

In [ ]:
# Sign in to Google and connect to the Disney dataset.
# Run this once per Colab session.

from google.colab import auth
auth.authenticate_user()

from google.cloud.bigquery import magics
magics.context.project = "sql-with-disney"

print("✓ Ready! Run the query cells below.")

## The point of `GROUP BY`

In Lesson 04 you ran:

```sql
SELECT COUNT(*) FROM disney_lessons.movies
```

…and got back **35** — one number for the whole table.

But what if you wanted *one number per studio* — how many Disney Animation movies? How many Pixar? How many Marvel? You could write four separate queries with different `WHERE` clauses, but that's tedious.

`GROUP BY` does it in one query: *"split the table into groups, then compute the aggregate within each group."*

## Movies per studio

The shape:

```
SELECT [grouping column], COUNT(*)
FROM [table]
GROUP BY [grouping column]
```

In English: *"split the table by studio, then count the rows in each group."*

In [ ]:
%%bigquery
SELECT studio, COUNT(*) AS movie_count
FROM disney_lessons.movies
GROUP BY studio

**4 rows** — one per studio. Each row tells you a studio name and how many movies it has in our table.

## The rule for `GROUP BY`

Every column in your `SELECT` must either:
1. Be in the `GROUP BY` clause, or
2. Be wrapped in an aggregate function (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`)

In our query above, `studio` is in `GROUP BY`, and `COUNT(*)` is an aggregate. ✅

If you tried `SELECT studio, title, COUNT(*) FROM ... GROUP BY studio`, BigQuery would error out — `title` is neither in the group nor aggregated. Why? Because there are 4 studio groups but 35 different titles; SQL doesn't know which title to show on the row for "Pixar" (there are 9!).

This rule is one of the most common stumbling blocks. Memorize it: **every selected column is either grouped or aggregated.**

## Other aggregates per group

`COUNT(*)` is just one option. You can use any aggregate per group.

Total box office per studio:

In [ ]:
%%bigquery
SELECT
  studio,
  ROUND(SUM(box_office_millions), 0) AS total_box_office_millions
FROM disney_lessons.movies
GROUP BY studio

Average score per genre:

In [ ]:
%%bigquery
SELECT
  genre,
  ROUND(AVG(rt_score), 1) AS avg_score
FROM disney_lessons.movies
GROUP BY genre

Multiple aggregates in one query:

In [ ]:
%%bigquery
SELECT
  studio,
  COUNT(*) AS movie_count,
  ROUND(AVG(rt_score), 1) AS avg_score,
  MAX(box_office_millions) AS biggest_hit
FROM disney_lessons.movies
GROUP BY studio

Now you have a real summary table — for each studio, how many movies they have, their average score, and their biggest box office number.

## `ORDER BY` works on the grouped result

You can sort the grouped output. Studios ranked by total box office, biggest first:

In [ ]:
%%bigquery
SELECT
  studio,
  ROUND(SUM(box_office_millions), 0) AS total_box_office
FROM disney_lessons.movies
GROUP BY studio
ORDER BY total_box_office DESC

Note: the `ORDER BY` references `total_box_office` — the alias you defined. That works in GoogleSQL. (Some other dialects make you repeat the full expression — `ORDER BY SUM(box_office_millions) DESC`. GoogleSQL accepts either.)

## Grouping by multiple columns

You can group by more than one column. The aggregate runs once per *combination* of values.

Movies per (studio, genre):

In [ ]:
%%bigquery
SELECT
  studio,
  genre,
  COUNT(*) AS movie_count
FROM disney_lessons.movies
GROUP BY studio, genre
ORDER BY studio, genre

Notice some studios show up twice (Marvel has both Action movies, none Animation; Pixar is all Animation). The combination is what defines a group.

## `HAVING` — filtering on aggregates

`WHERE` filters **rows before grouping**. `HAVING` filters **groups after aggregation**. They're different — and the difference matters.

Show me studios with **more than 10 movies** in our table:

In [ ]:
%%bigquery
SELECT
  studio,
  COUNT(*) AS movie_count
FROM disney_lessons.movies
GROUP BY studio
HAVING COUNT(*) > 10

You can't put `COUNT(*) > 10` in `WHERE` — `WHERE` doesn't know about aggregates yet (the grouping hasn't happened). `HAVING` is `WHERE` for groups.

## `WHERE` and `HAVING` together

A common pattern: filter rows with `WHERE`, group, then filter groups with `HAVING`.

Studios with average box office over $500M, **only counting movies released after 2000**:

In [ ]:
%%bigquery
SELECT
  studio,
  COUNT(*) AS recent_movies,
  ROUND(AVG(box_office_millions), 0) AS avg_box_office
FROM disney_lessons.movies
WHERE year > 2000
GROUP BY studio
HAVING AVG(box_office_millions) > 500
ORDER BY avg_box_office DESC

The query order in your head:
1. `WHERE year > 2000` — keep only movies from after 2000
2. `GROUP BY studio` — group those filtered rows by studio
3. Compute aggregates per studio
4. `HAVING AVG(...) > 500` — keep only studios whose group average is above 500
5. `ORDER BY` — sort the final groups

## Quick reminders

- `WHERE` filters rows. `HAVING` filters groups. They're not interchangeable.
- Every selected column must be grouped or aggregated.
- Read-only as always. `%%bigquery` is still just Colab plumbing.

## Exercises

### Exercise 1

For each **genre**, how many movies are there? Show `genre` and `movie_count`, sorted with the most-common genre first.

In [ ]:
# Your query here

### Exercise 2

Average **runtime** per studio. Show `studio` and `avg_runtime` (rounded to 1 decimal), sorted longest-first.

In [ ]:
# Your query here

### Exercise 3

For each **year**, how many movies were released? Only show years that had **2 or more movies**. Sort by year ascending.

Hint: `GROUP BY year` and use `HAVING` to filter.

In [ ]:
# Your query here

### Exercise 4 (bonus)

Per studio, what's the **highest-grossing movie's box office** (`MAX(box_office_millions)`)? Sort biggest first.

In [ ]:
# Your query here

---

## Solutions

⚠️ **Spoilers.**

### Solution 1

In [ ]:
%%bigquery
SELECT
  genre,
  COUNT(*) AS movie_count
FROM disney_lessons.movies
GROUP BY genre
ORDER BY movie_count DESC

### Solution 2

In [ ]:
%%bigquery
SELECT
  studio,
  ROUND(AVG(runtime_minutes), 1) AS avg_runtime
FROM disney_lessons.movies
GROUP BY studio
ORDER BY avg_runtime DESC

### Solution 3

In [ ]:
%%bigquery
SELECT
  year,
  COUNT(*) AS movies_released
FROM disney_lessons.movies
GROUP BY year
HAVING COUNT(*) >= 2
ORDER BY year

### Solution 4

In [ ]:
%%bigquery
SELECT
  studio,
  MAX(box_office_millions) AS biggest_hit_millions
FROM disney_lessons.movies
GROUP BY studio
ORDER BY biggest_hit_millions DESC

## What you've learned

- ✅ `GROUP BY` to compute one aggregate per group
- ✅ The selected-columns rule: grouped or aggregated, no other option
- ✅ Multi-column grouping (one row per combination)
- ✅ `HAVING` to filter on the aggregated result
- ✅ Combining `WHERE` (rows) + `GROUP BY` + `HAVING` (groups)

## Up next

You've been working with one table at a time. But real data is spread across many tables — you usually need information from a *combination* of them. That's **joining**, and it unlocks pretty much everything.

Lesson 06 introduces the `characters` table and shows you how to combine it with `movies`. Open `06_joins.ipynb`.